## Building a chatbot using Langchain


##### Basically in this file I will be going to design and implement an LLM powered Chatbot. This will be able to have a conversation and remember previous interactions as well.


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
groq_api_key=os.getenv("GROQ_API_KEY")

In [3]:
from langchain_groq import ChatGroq
model = ChatGroq(model="openai/gpt-oss-120b",groq_api_key=groq_api_key)

In [4]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, this is Manish Sabbani, I am Data Engineer with AI/ML Background")])

AIMessage(content='Hello Manish! 👋 Great to meet you. As a Data Engineer with an AI/ML background, you’ve got a powerful blend of skills. How can I assist you today? Whether you’d like to discuss architecture design, data pipelines, model deployment strategies, career advice, or anything else, just let me know!', additional_kwargs={'reasoning_content': 'The user just says "Hi, this is Manish Sabbani, I am Data Engineer with AI/ML Background". Likely they want a response. As ChatGPT, we should greet them, ask how we can help, maybe discuss their background. There\'s no policy violation. So respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 136, 'prompt_tokens': 89, 'total_tokens': 225, 'completion_time': 0.294370737, 'completion_tokens_details': {'reasoning_tokens': 61}, 'prompt_time': 0.00344543, 'prompt_tokens_details': None, 'queue_time': 0.005069097, 'total_time': 0.297816167}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_1d982b31b2', 'ser

In [6]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi, this is Manish Sabbani, I am Data Engineer with AI/ML Background"),
        AIMessage(content="Hello Manish! 👋 Great to meet you. As a Data Engineer with an AI/ML background, you’ve got a powerful blend of skills. How can I assist you today?"),
        HumanMessage(content="What is my designation? What I do? And what is my name")
    ]
)

AIMessage(content='**Name:**\u202fManish\u202fSabbani  \n**Designation:**\u202fData Engineer (with an AI/ML background)  \n**What you do:**\u202fYou design, build, and maintain robust data pipelines and infrastructure that enable reliable data collection, storage, and processing. Leveraging your AI/ML expertise, you also prepare data for model training, support feature engineering, and help integrate machine‑learning models into production systems, ensuring that data flows smoothly from source to insight.', additional_kwargs={'reasoning_content': 'The user asks: "What is my designation? What I do? And what is my name". The user earlier introduced themselves: "Hi, this is Manish Sabbani, I am Data Engineer with AI/ML Background". So we answer: designation: Data Engineer (with AI/ML background). What they do: work on data pipelines, ETL, data modeling, integration, support AI/ML workflows, etc. Name: Manish Sabbani. Provide concise answer.'}, response_metadata={'token_usage': {'completio

#### Message History

In [7]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [10]:
store = {}
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [9]:
config = {"configurable":{"session_id":"chat1"}}


In [12]:
response=with_message_history.invoke(
    [
        HumanMessage(content="Hi I am Manish Sabbani, I am a data engineer")
    ],
    config=config
)

In [13]:
response.content

'Nice to meet you, Manish! 👋 As a data engineer, you’re probably juggling everything from data ingestion to transformation, storage, and orchestration. \n\nWhat’s on your mind today?\n\n- Designing or refactoring an ETL/ELT pipeline?  \n- Choosing the right data lake/warehouse technology?  \n- Optimizing Spark, Flink, or SQL queries?  \n- Setting up CI/CD, monitoring, or data quality checks?  \n- Anything else data‑engineering‑related?\n\nLet me know the details, and I’ll dive right in!'

In [15]:
response2=with_message_history.invoke(
    [
        HumanMessage(content="Hi what is my name?")
    ],config=config
)

In [18]:
# change the session_id
config1 = {"configurable":{"session_id":"chat1"}}
response3=with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config1
)

In [19]:
response3.content

'Your name is **Manish\u202fSabbani**.'

In [20]:
response3=with_message_history.invoke(
    [HumanMessage(content="Hi my name is sahithi.")],
    config=config1
)

In [21]:
response3=with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config1
)

In [22]:
response3.content

'Your name is **Sahithi**.'

#### Prompt Templates


In [24]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
generic_prompt = ChatPromptTemplate.from_messages(
    [
       ("system"," You are my helpful assistant, answer all the questions to the best of your ability"),
       MessagesPlaceholder(variable_name="messages")
    ]
)

chain = generic_prompt|model

In [26]:
chain.invoke({"messages":[HumanMessage(content="Hi this is Sahiman")]})

AIMessage(content='Hello, Sahiman! Nice to meet you. How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "Hi this is Sahiman". Likely a greeting. We respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 95, 'total_tokens': 141, 'completion_time': 0.101979008, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.00447442, 'prompt_tokens_details': None, 'queue_time': 0.04426624, 'total_time': 0.106453428}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4140daa9c2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019bbf80-382f-7d32-bcac-945cdbbdfcb3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 95, 'output_tokens': 46, 'total_tokens': 141, 'output_token_details': {'reasoning': 20}})

In [27]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [ ]:
config = {"configurable":{"session_id":"chat3"}}
response=with_message_history.invoke([HumanMessage(content="Hi this is Sahiman")],
                                     config=config)
response.content

In [29]:
generic_prompt = ChatPromptTemplate.from_messages(
    [
       ("system"," You are my helpful assistant, answer all the questions to the best of your ability in {language}."),
       MessagesPlaceholder(variable_name="messages")
    ]
)

chain = generic_prompt|model

In [30]:
response=chain.invoke({"messages":[HumanMessage(content="Hi this is Sahiman")],"language":"Telugu"})
response.content

'హలో, సహిమన్! మీరు ఎలా ఉన్నారు? మీకు ఏదైనా సహాయం కావాలా? 😊'

In [31]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history,input_messages_key="messages")

In [32]:
config={"configurable":{"session_id":"chat4"}}
response=with_message_history.invoke(
    {'messages':[HumanMessage(content="What is my name")], "language":"telugu"},
    config=config
)

In [33]:
response.content

'క్షమించండి, మీ పేరు నాకు తెలియదు. మీరు మీ పేరును చెప్పితే, దానిని గుర్తుంచుకుని తరువాతి సంభాషణల్లో ఉపయోగించగలను.'

In [36]:
config={"configurable":{"session_id":"chat4"}}
response=with_message_history.invoke(
    {'messages':[HumanMessage(content="What is my name")], "language":"telugu"},
    config=config
)

In [37]:
response.content

'క్షమించండి, మీ పేరు నాకు తెలియదు. మీరు మీ పేరును చెప్పగలరా? దాన్ని తెలుసుకుంటే, తదుపరి సంభాషణల్లో దాన్ని ఉపయోగించగలను.'

In [38]:
config={"configurable":{"session_id":"chat4"}}
response=with_message_history.invoke(
    {'messages':[HumanMessage(content=" my name is Manish")], "language":"telugu"},
    config=config
)

In [39]:
response.content

'సంతోషంగా మీను కలుసుకున్నాను, మనీష్! మీ పేరు ఇప్పుడు నాకు తెలుసు, తదుపరి సంభాషణల్లో మీ పేరుతోనే సంభాషిస్తాను. 😊'

In [40]:
config={"configurable":{"session_id":"chat4"}}
response=with_message_history.invoke(
    {'messages':[HumanMessage(content=" now tell me what is my name?")], "language":"telugu"},
    config=config
)

In [41]:
response.content

'మీ పేరు\u202f**మనీష్**.'

In [42]:
config={"configurable":{"session_id":"chat4"}}
response=with_message_history.invoke(
    {'messages':[HumanMessage(content=" Do you remember my name??")], "language":"telugu"},
    config=config
)

In [43]:
response.content

'అవును, నేను మీ పేరును గుర్తు పెట్టుకున్నాను — మీ పేరు **మనీష్**. మరేదైనా సహాయం కావాలంటే చెప్పండి!'

#### Managing the Chat/Conversation History

In [44]:
from langchain_core.messages import HumanMessage, SystemMessage, trim_messages

In [46]:
trimmer=trim_messages(
    max_tokens=70,
    strategy="last",
    token_counter=model,
    include_system=True,allow_partial=False,start_on="human"
)

# we will just add some set of messages
messages = [
    SystemMessage(content="You are a good assistant"),
    HumanMessage(content="Hi, I am Manish Kumar"),
    AIMessage(content="Hello!!!"),
    HumanMessage(content="I Like to play Cricket"),
    SystemMessage(content="That's nice"),
    SystemMessage(content="You are an encouraging assistant."),
    HumanMessage(content="I feel stuck in my work."),
    AIMessage(content="That happens sometimes. Taking a short break or changing perspective can help."),
    HumanMessage(content="Any advice to stay motivated?"),
    AIMessage(content="Focus on progress, not perfection, and celebrate small wins."),
    SystemMessage(content="You are an assistant that engages users politely."),
    HumanMessage(content="What are some good hobbies to pick up?"),
    AIMessage(content="Reading, playing sports, learning music, or even coding can be great hobbies."),
    HumanMessage(content="I like outdoor activities."),
    AIMessage(content="That’s great! Hiking, cycling, or playing sports like cricket are excellent choices.")

]

In [ ]:
trimmer.invoke(messages)

In [51]:
#applying the trimmer
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    |generic_prompt
    |model
    
)


latest_response=chain.invoke(
    {"messages": messages + [HumanMessage(content="What is my favourite game")],
             "language":"telugu"})
latest_response.content



'మీ ఇష్టమైన ఆట గురించి మీరు ఇప్పటివరకు చెప్పలేదు, కాబట్టి దయచేసి మీరు ఏ ఆటను ఇష్టపడుతున్నారో చెప్పగలరా? మీరు చెప్పిన తర్వాత, దానికి సంబంధించిన సమాచారం లేదా సూచనలు ఇవ్వడానికి నేను సంతోషిస్తాను!'

In [54]:
# Wrapping it in the message history

with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)
config={"configurable":{"session_id":"chat5"}}


In [55]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    |generic_prompt
    |model
    
)


latest_response=chain.invoke(
    {"messages": messages + [HumanMessage(content="Cricket is my favourite game")],
             "language":"telugu"})
latest_response.content

'అద్భుతం! క్రికెట్ నిజంగా ఉత్సాహభరితమైన ఆట. మీకు ఇష్టమైన జట్టు, ఆటగాడు లేదా మీరు ఎక్కువగా అనుసరించే టోర్నమెంట్ ఏది? మీకు ఇష్టమైన మ్యాచ్ లేదా గుర్తుండిపోయే ఒక క్షణం ఉంటే, దాన్ని కూడా పంచుకోవచ్చు! 🎉🏏'

In [56]:
chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    |generic_prompt
    |model
    
)


latest_response=chain.invoke(
    {"messages": messages + [HumanMessage(content="now tell me What is my favourite game")],
             "language":"telugu"})
latest_response.content

'మీకు ఇష్టమైన ఆట ఏది అనేది నాకు తెలియదు. మీరు ఏ ఆటను ఎక్కువగా ఆస్వాదిస్తారో చెప్పగలరా?'

In [57]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)
config={"configurable":{"session_id":"chat5"}}

In [58]:
latest_response=chain.invoke(
    {"messages": messages + [HumanMessage(content="now tell me What is my favourite game")],
             "language":"telugu"})

In [59]:
latest_response.content

'మీకు ఏ ఆట ఎక్కువగా నచ్చుతుందో నేను ఖచ్చితంగా చెప్పలేను, ఎందుకంటే మీరు ఇంకా దాని గురించి చెప్పలేదు. మీరు మీకు ఇష్టమైన ఆట ఏది అని చెప్పగలరా? దాని గురించి తెలుసుకుంటే, నేను మీకు మరింత సమాచారం లేదా సంబంధిత సూచనలను ఇవ్వగలను.'

In [60]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)
config={"configurable":{"session_id":"chat5"}}

In [64]:
latest_response=chain.invoke(
    {"messages": messages + [HumanMessage(content="Cricket is my favourite game")],
             "language":"English"})
latest_response.content

'Awesome! Cricket is such a fun blend of strategy, skill, and teamwork. Do you have a favorite format—Test, One\u202fDay, or T20? And are you more into batting, bowling, fielding, or just enjoying the game as a fan? If you ever want ideas for cricket‑related outdoor activities (like net practice, backyard games, or even a friendly match with friends), just let me know!'

In [65]:
latest_response=chain.invoke(
    {"messages": messages + [HumanMessage(content="now tell me What is my favourite game")],
             "language":"English"})
latest_response.content

'I’m not sure which game you enjoy most—could you share a bit more about the sports or games you like?'

In [66]:
from langchain_core.messages import HumanMessage, AIMessage

# user says something
messages.append(HumanMessage(content="Cricket is my favourite game"))

latest_response = chain.invoke({
    "messages": messages,
    "language": "English"
})

# SAVE THE AI RESPONSE
messages.append(AIMessage(content=latest_response.content))


In [67]:
messages.append(
    HumanMessage(content="now tell me what is my favourite game")
)

latest_response = chain.invoke({
    "messages": messages,
    "language": "English"
})

messages.append(AIMessage(content=latest_response.content))

print(latest_response.content)


I’m not sure what your favorite game is—could you give me a hint or tell me a bit about the kinds of games you enjoy? That way I can try to guess or suggest something that might be close!
